## Regressão polinomial — `valor_aluguel ~ distancia_praia_km` (vários graus)

Mesma base (`bases/aluguel_praia_polinomial.csv`) e a mesma dupla `distancia_praia_km`
→ `valor_aluguel` do notebook de regressão linear simples — mas agora testamos
se **transformar** o atributo (`distancia²`, `distancia³`, ...) melhora o
ajuste. Regressão polinomial não é um algoritmo novo: continua sendo
`LinearRegression`, só que treinada sobre atributos transformados.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures

SEMENTE = 42
pd.set_option("display.width", 100)

df = pd.read_csv("bases/aluguel_praia_2.csv")
X = df[["distancia_praia_km"]].values
y = df["valor_aluguel"].values

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.30, random_state=SEMENTE
)
print(f"treino: {X_treino.shape[0]} imoveis   teste: {X_teste.shape[0]} imoveis")

FileNotFoundError: [Errno 2] No such file or directory: 'bases/aluguel_praia_2.csv'

### 1. Por que suspeitar de uma reta aqui

O gráfico de dispersão de `distancia_praia_km` x `valor_aluguel` já mostra, a
olho nu, uma curva de decaimento — não uma reta: o aluguel cai rápido nos
primeiros quilômetros e depois estabiliza num patamar.

### 2. Ajustando vários graus e comparando treino x teste

A cada grau, `PolynomialFeatures` cria as colunas extras (`distancia²`,
`distancia³`, ...) e uma `LinearRegression` comum é treinada sobre elas.
Medimos R² tanto no treino quanto no teste — a distância entre os dois é o
sinal prático de overfitting.

In [ ]:
graus = [1, 2, 3, 4, 6, 8]
linhas = []

for grau in graus:
    poly = PolynomialFeatures(degree=grau, include_bias=False)
    Xtr_p = poly.fit_transform(X_treino)
    Xte_p = poly.transform(X_teste)

    modelo = LinearRegression().fit(Xtr_p, y_treino)
    pred_tr = modelo.predict(Xtr_p)
    pred_te = modelo.predict(Xte_p)

    linhas.append({
        "grau": grau,
        "R2_treino": r2_score(y_treino, pred_tr),
        "R2_teste": r2_score(y_teste, pred_te),
        "MAE_teste": mean_absolute_error(y_teste, pred_te),
        "RMSE_teste": mean_squared_error(y_teste, pred_te) ** 0.5,
    })

tabela_graus = pd.DataFrame(linhas)
tabela_graus["gap_treino_menos_teste"] = tabela_graus["R2_treino"] - tabela_graus["R2_teste"]
tabela_graus.round(4)

**Leitura:** o R² de teste sobe bastante do grau 1 para o grau 2 (a curva
sendo capturada) e continua melhorando um pouco até o grau 3 ou 4 — depois disso
o ganho fica pequeno, e o `gap` entre R² de treino e de teste tende a crescer
nos graus mais altos, o sinal prático de overfitting.

### 3. Visualizando as curvas

In [ ]:
grade = np.linspace(df["distancia_praia_km"].min(), df["distancia_praia_km"].max(), 300).reshape(-1, 1)
cores = {1: "#c0392b", 2: "#1f6f43", 4: "#2c5aa0"}

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.scatter(X_teste, y_teste, s=14, alpha=0.35, color="#8a8a8a", label="dados de teste")

for grau, cor in cores.items():
    poly = PolynomialFeatures(degree=grau, include_bias=False)
    Xtr_p = poly.fit_transform(X_treino)
    modelo = LinearRegression().fit(Xtr_p, y_treino)
    y_grade = modelo.predict(poly.transform(grade))
    estilo = "--" if grau == 1 else "-"
    ax.plot(grade, y_grade, color=cor, linewidth=2.2, linestyle=estilo, label=f"grau {grau}")

ax.set_xlabel("distancia da praia (km)")
ax.set_ylabel("valor do aluguel (R$)")
ax.set_title("Regressao polinomial: efeito do grau sobre a mesma base")
ax.legend(frameon=False, fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

### 4. Escolhendo um grau e treinando o modelo final

Com base na tabela da Seção 2, o grau 3 é um bom equilíbrio: captura a maior
parte do ganho disponível sem um `gap` treino-teste muito maior que o do grau 2.

In [ ]:
GRAU_ESCOLHIDO = 3
poly_final = PolynomialFeatures(degree=GRAU_ESCOLHIDO, include_bias=False)
X_treino_poly = poly_final.fit_transform(X_treino)
modelo_final = LinearRegression().fit(X_treino_poly, y_treino)

pred_final = modelo_final.predict(poly_final.transform(X_teste))
print(f"grau escolhido = {GRAU_ESCOLHIDO}")
print(f"R^2 (teste)   = {r2_score(y_teste, pred_final):.4f}")
print(f"MAE (teste)   = R$ {mean_absolute_error(y_teste, pred_final):.2f}")
print(f"RMSE (teste)  = R$ {mean_squared_error(y_teste, pred_final) ** 0.5:.2f}")

### 5. Pesquisar um valor

Digite uma distância até a praia (em km) e o modelo polinomial (grau escolhido
acima) devolve o aluguel previsto.

In [ ]:
def prever_valor_aluguel(distancia_km):
    X_novo = poly_final.transform(np.array([[distancia_km]]))
    return modelo_final.predict(X_novo)[0]


try:
    entrada = input("Digite a distancia da praia em km: ")
    distancia_pesquisada = float(entrada.replace(",", "."))
except Exception:
    distancia_pesquisada = 3.0
    print(f"(entrada interativa indisponivel - usando valor de exemplo: {distancia_pesquisada} km)")

valor_previsto = prever_valor_aluguel(distancia_pesquisada)
print(f"\nDistancia pesquisada: {distancia_pesquisada:.2f} km")
print(f"Valor de aluguel previsto (regressao polinomial, grau {GRAU_ESCOLHIDO}): R$ {valor_previsto:.2f}")

### O que guardar deste notebook

- Regressão polinomial é regressão linear sobre atributos transformados (x, x²,
  x³, ...) — não é um algoritmo diferente, é uma mudança no `X`.
- Grau maior sempre reduz (ou mantém) o erro de TREINO — por isso nunca se
  escolhe o grau olhando só para o treino; o R² de TESTE é que mostra onde o
  ganho para.
- Comparado ao notebook de regressão múltipla (que usa três atributos), este
  modelo usa só `distancia_praia_km` e ainda assim tende a se sair melhor —
  porque o que faltava não era mais informação, era a forma certa para o
  efeito da distância.